# 02 · Comparar costas dentro de un rango de ingreso · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

La base equivale a `df` del notebook original: dormitorios imputados, marca de imputación y tres ratios. No se aplicó el filtro de curado.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_viviendas()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 20,640 | Columnas: 14


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,False,6.281853,0.172096,2.181467


## 2. Consigna y resultado esperado

Filtrá las zonas cuyo `median_income` esté en el intervalo
**`ingreso_min <= ingreso < ingreso_max`** y resumí cada categoría de costa.

Devolvé un DataFrame con índice `ocean_proximity`, ordenado alfabéticamente,
y estas columnas, en este orden:

| Columna | Cálculo |
|---|---|
| `zonas` | Cantidad de filas del grupo |
| `precio_mediano` | Mediana de `median_house_value` |
| `ingreso_mediano` | Mediana de `median_income` |
| `habitaciones_por_hogar_mediana` | Mediana de `rooms_per_household` |

Incluí solo categorías presentes después del filtro. Si ninguna fila cumple,
devolvé una tabla vacía con ese índice y esas columnas. No redondees ni modifiques la entrada.
Podés asumir límites numéricos con `ingreso_min < ingreso_max`.

## 3. Resolución comentada

1. La máscara combina `>= ingreso_min` y `< ingreso_max` con `&`.
2. `groupby` trabaja sobre las filas que quedaron después del filtro.
3. `size` cuenta zonas y `median` resume las otras variables; `agg` asigna los nombres pedidos.

`sort=True` ordena las costas. `observed=True` evita agregar categorías sin filas cuando
la columna es categórica. Una selección vacía conserva la estructura de la salida.

In [3]:
def resumir_por_costa(datos, ingreso_min=3, ingreso_max=5):
    """Resume las zonas de cada costa dentro del intervalo de ingreso solicitado."""
    # El límite inferior se incluye; el superior queda fuera del filtro.
    en_rango = (
        datos["median_income"].ge(ingreso_min)
        & datos["median_income"].lt(ingreso_max)
    )
    filtrado = datos.loc[en_rango]

    # Las agregaciones nombradas definen el cálculo y la columna de salida.
    return filtrado.groupby("ocean_proximity", sort=True, observed=True).agg(
        zonas=("median_house_value", "size"),
        precio_mediano=("median_house_value", "median"),
        ingreso_mediano=("median_income", "median"),
        habitaciones_por_hogar_mediana=("rooms_per_household", "median"),
    )

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "ocean_proximity": ["B", "A", "B", "A", "C", "A", "B"],
        "median_income": [2.9, 3, 4, 5, 3, 4.9, 7],
        "median_house_value": [500, 100, 200, 900, 80, 300, 700],
        "rooms_per_household": [8, 2, 4, 9, 1, 6, 7],
    })
    original = caso.copy(deep=True)
    columnas = ["zonas", "precio_mediano", "ingreso_mediano", "habitaciones_por_hogar_mediana"]
    esperado = pd.DataFrame(
        [[2, 200, 3.95, 4], [1, 200, 4, 4], [1, 80, 3, 1]],
        columns=columnas, index=pd.Index(["A", "B", "C"], name="ocean_proximity"),
    )
    comparar(funcion(caso), esperado)

    otro_rango = pd.DataFrame(
        [[1, 300, 4.9, 6], [1, 200, 4, 4]],
        columns=columnas, index=pd.Index(["A", "B"], name="ocean_proximity"),
    )
    comparar(funcion(caso, ingreso_min=4, ingreso_max=5), otro_rango)
    comparar(funcion(caso, ingreso_min=20, ingreso_max=30), esperado.iloc[:0])
    comparar(funcion(caso.iloc[:0]), esperado.iloc[:0])
    comparar(caso, original)
    print("OK: límites inclusivo/exclusivo, medianas, orden, otro rango y resultado vacío.")


probar(resumir_por_costa)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = resumir_por_costa(datos)
comparar(datos, antes)
filtro = datos["median_income"].ge(3) & datos["median_income"].lt(5)
assert resultado["zonas"].sum() == filtro.sum()
assert resultado.index.is_monotonic_increasing
assert resultado["ingreso_mediano"].ge(3).all() and resultado["ingreso_mediano"].lt(5).all()
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: límites inclusivo/exclusivo, medianas, orden, otro rango y resultado vacío.
OK: dataset real. Resultado: 5 filas.


,zonas,precio_mediano,ingreso_mediano,habitaciones_por_hogar_mediana
ocean_proximity,,,,
<1H OCEAN,4016,202800.0,3.85710,4.997303
INLAND,2573,129200.0,3.68520,5.767141
ISLAND,1,300000.0,3.39060,7.385417
NEAR BAY,1012,229350.0,3.88425,5.010182
NEAR OCEAN,1164,237500.0,3.85765,5.166977


## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.